# Recommender Systems: Two-Tower, TFRS & Faiss ANN — Module 4 Specialization

## Learning Objectives
- **Two-Tower Architecture**: Separate user/item encoders with dot-product similarity
- **TensorFlow Recommenders (TFRS)**: End-to-end retrieval + ranking pipeline
- **Faiss ANN**: Approximate Nearest Neighbor search for scalable retrieval
- **Implicit Feedback**: Matrix factorization, BPR loss, weighted ALS
- **Cold Start**: Content-based features, meta-learning, popularity fallback
- **Evaluation**: Recall@K, NDCG@K, MAP, coverage, diversity

## Why This Matters
Recommender systems power discovery at Netflix, YouTube, Amazon, Spotify, TikTok. ML Architects must design two-stage systems: **retrieval** (candidate generation from millions) → **ranking** (precise scoring of hundreds). This notebook implements the full pipeline with real training and Faiss-based serving.

In [ ]:
# Install required packages
import sys
!{sys.executable} -m pip install tensorflow-recommenders faiss-cpu pandas numpy matplotlib scikit-learn -q 2>/dev/null || true

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_recommenders as tfrs
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
import faiss

tf.random.set_seed(42)
np.random.seed(42)

print(f\"TensorFlow: {tf.__version__}\")
print(f\"TFRS: {tfrs.__version__}\")
print(f\"Faiss: {faiss.__version__}\")

## 1. Synthetic Dataset: MovieLens-Style Interactions

Generate realistic implicit feedback data with user/item features for content-based cold start.

In [ ]:
# Generate synthetic data
n_users = 5000
n_items = 2000
n_interactions = 100000

# User features
user_ids = np.arange(n_users)
user_age = np.random.randint(18, 70, n_users)
user_gender = np.random.randint(0, 2, n_users)  # 0=F, 1=M
user_genre_prefs = np.random.dirichlet(np.ones(10), n_users)  # 10 genres

# Item features
item_ids = np.arange(n_items)
item_year = np.random.randint(1990, 2024, n_items)
item_genres = np.random.dirichlet(np.ones(10), n_items)  # genre distribution
item_popularity = np.random.zipf(2, n_items)  # long-tail popularity
item_popularity = item_popularity / item_popularity.max() * 100

# Generate interactions (implicit feedback: watch, click, purchase)
# Higher probability for items matching user genre preferences
interactions = []
for _ in range(n_interactions):
    u = np.random.randint(0, n_users)
    # Sample item weighted by popularity + preference match
    prefs = user_genre_prefs[u]
    match_scores = item_genres @ prefs
    probs = (item_popularity + 1) * (match_scores + 0.1)
    probs = probs / probs.sum()
    i = np.random.choice(n_items, p=probs)
    interactions.append((u, i))

interactions = np.array(interactions)
interactions_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])
interactions_df['timestamp'] = np.random.randint(1609459200, 1704067200, n_interactions)
interactions_df = interactions_df.drop_duplicates(subset=['user_id', 'item_id'])

print(f\"Unique users: {interactions_df['user_id'].nunique()}\")
print(f\"Unique items: {interactions_df['item_id'].nunique()}\")
print(f\"Interactions: {len(interactions_df)}\")
print(f\"Sparsity: {1 - len(interactions_df) / (n_users * n_items):.4f}\")

## 2. Two-Tower Model Architecture

```
User Tower                    Item Tower
┌─────────┐                   ┌─────────┐
│ User ID │                   │ Item ID │
└────┬────┘                   └────┬────┘
     │                             │
┌────▼────┐                   ┌────▼────┐
│Embedding│                   │Embedding│
└────┬────┘                   └────┬────┘
     │                             │
┌────▼────┐                   ┌────▼────┐
│  Dense  │                   │  Dense  │
│ Layers  │                   │ Layers  │
└────┬────┘                   └────┬────┘
     │                             │
┌────▼────┐                   ┌────▼────┐
│ User    │                   │ Item    │
│ Vector  │                   │ Vector  │
│  (64)   │                   │  (64)   │
└────┬────┘                   └────┬────┘
      \\                           /
       \\                         /
        \\                       /
         ┌───────────────────┐
         │   Dot Product     │
         │   (Similarity)    │
         └───────────────────┘
```

In [ ]:
# Prepare TFRS dataset
ratings = tf.data.Dataset.from_tensor_slices({
    'user_id': interactions_df['user_id'].values.astype(np.int64),
    'item_id': interactions_df['item_id'].values.astype(np.int64),
    'timestamp': interactions_df['timestamp'].values.astype(np.int64)
})

# Add user/item features
users = tf.data.Dataset.from_tensor_slices({
    'user_id': user_ids.astype(np.int64),
    'age': user_age.astype(np.float32),
    'gender': user_gender.astype(np.int64),
    'genre_prefs': user_genre_prefs.astype(np.float32)
})

items = tf.data.Dataset.from_tensor_slices({
    'item_id': item_ids.astype(np.int64),
    'year': item_year.astype(np.float32),
    'genres': item_genres.astype(np.float32),
    'popularity': item_popularity.astype(np.float32)
})

# Vocabulary for StringLookup
user_ids_vocab = user_ids.astype(str)
item_ids_vocab = item_ids.astype(str)

# Shuffle and split
shuffled = ratings.shuffle(100000, seed=42, reshuffle_each_iteration=False)
train = shuffled.take(80000)
test = shuffled.skip(80000).take(20000)

# Candidate set for retrieval (all items)
candidates = items.batch(128)

print(\"Datasets prepared.\")

## 3. Retrieval Model (Two-Tower) with TFRS

In [ ]:
# User model
class UserModel(tf.keras.Model):
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=user_ids_vocab, mask_token=None),
            tf.keras.layers.Embedding(len(user_ids_vocab) + 1, embedding_dim)
        ])
        self.age_norm = tf.keras.layers.Normalization()
        self.age_norm.adapt(users.map(lambda x: x['age']))
        self.gender_embedding = tf.keras.Sequential([
            tf.keras.layers.Embedding(2, 8)
        ])
        self.genre_dense = tf.keras.layers.Dense(32, activation='relu')
        
        self.dense_layers = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(embedding_dim)
        ])
    
    def call(self, inputs):
        user_id_emb = self.user_embedding(inputs['user_id'])
        age_norm = tf.expand_dims(self.age_norm(inputs['age']), axis=1)
        gender_emb = self.gender_embedding(inputs['gender'])
        genre_emb = self.genre_dense(inputs['genre_prefs'])
        
        # Concatenate all features


In [ ]:
# TFRS Retrieval Task
class RetrievalModel(tfrs.Model):
    def __init__(self, user_model, item_model):
        super().__init__()
        self.user_model = user_model
        self.item_model = item_model
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=candidates.map(item_model)
            )
        )
    
    def compute_loss(self, features, training=False):
        user_emb = self.user_model({


## 4. Faiss Approximate Nearest Neighbor (ANN) Index

For production serving, we need sub-millisecond retrieval from millions of items. Faiss provides GPU-accelerated ANN search.

In [ ]:
# Generate all item embeddings
print(\"Generating item embeddings for Faiss index...\")
all_item_embs = []
for batch in items.batch(512):
    emb = item_model(batch).numpy()
    all_item_embs.append(emb)

all_item_embs = np.vstack(all_item_embs).astype(np.float32)
print(f\"Item embeddings shape: {all_item_embs.shape}\")

# Normalize for cosine similarity (dot product on normalized = cosine)
faiss.normalize_L2(all_item_embs)

# Build Faiss index
# IndexFlatIP: exact inner product (brute force)
# IndexIVFFlat: inverted file with flat codes (approximate)
# IndexHNSW: hierarchical navigable small world (best recall/speed)

dim = all_item_embs.shape[1]

# Exact index (for comparison)
index_exact = faiss.IndexFlatIP(dim)
index_exact.add(all_item_embs)

# HNSW index (production choice)
index_hnsw = faiss.IndexHNSWFlat(dim, 32)  # M=32 connections
index_hnsw.hnsw.efConstruction = 80
index_hnsw.hnsw.efSearch = 64
index_hnsw.add(all_item_embs)

# IVF index (for very large scale)
nlist = 100
quantizer = faiss.IndexFlatIP(dim)
index_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(all_item_embs)
index_ivf.add(all_item_embs)
index_ivf.nprobe = 10

print(\"Faiss indices built: Exact, HNSW, IVF\")

In [ ]:
# Benchmark retrieval
import time

# Sample queries
n_queries = 100
query_users = np.random.choice(user_ids, n_queries)
query_embs = []
for u in query_users:
    u_batch = {
        'user_id': np.array([u]),


## 5. Ranking Model: Fine-Grained Scoring

Retrieval gives top-K candidates (e.g., 100-1000). Ranking model scores them precisely using richer features (cross features, interaction history).

In [ ]:
# Ranking model: takes user + item features, outputs score
class RankingModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        # User tower (shared with retrieval or separate)
        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=user_ids_vocab, mask_token=None),
            tf.keras.layers.Embedding(len(user_ids_vocab) + 1, 64)
        ])
        self.item_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=item_ids_vocab, mask_token=None),
            tf.keras.layers.Embedding(len(item_ids_vocab) + 1, 64)
        ])
        
        # Cross features
        self.cross_layer = tf.keras.layers.Dense(64, activation='relu')

        # Scoring head
        self.score_head = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(1)
        ])
    
    def call(self, inputs):
        user_emb = self.user_embedding(inputs['user_id'])
        item_emb = self.item_embedding(inputs['item_id'])
        
        # Element-wise product (cross)
        cross = user_emb * item_emb
        cross = self.cross_layer(cross)
        
        # Concat: user + item + cross


## 6. End-to-End Retrieval + Ranking Pipeline

In [ ]:
def recommend(user_id, k_retrieve=100, k_rank=10):
    \"\"\"Two-stage recommendation: retrieve -> rerank\"\"\"
    # Stage 1: Retrieval via Faiss HNSW
    u_batch = {
        'user_id': np.array([user_id]),


## 7. Evaluation: Recall@K, NDCG@K, Coverage, Diversity

In [ ]:
def evaluate_recommender(test_interactions, k_values=[5, 10, 20, 50]):
    \"\"\"Evaluate retrieval + ranking pipeline\"\"\"
    # Group test interactions by user
    test_by_user = test_interactions.groupby('user_id')['item_id'].apply(list).to_dict()


## 8. Cold Start Strategies

Handle new users/items without interaction history.

In [ ]:
# Cold-start: new user with only demographic/features
def cold_start_user_recommend(age, gender, genre_prefs, k=10):
    \"\"\"Recommend for new user using content-based features only\"\"\"
    # Create dummy user_id (will be mapped to OOV embedding)
    u_batch = {
        'user_id': np.array(['new_user']),


## 9. Implicit Feedback: BPR & Weighted ALS

Alternative approaches for implicit feedback (no explicit ratings).

In [ ]:
# Bayesian Personalized Ranking (BPR) loss implementation
class BPRModel(tf.keras.Model):
    def __init__(self, n_users, n_items, dim=64):
        super().__init__()
        self.user_emb = tf.keras.layers.Embedding(n_users, dim)
        self.item_emb = tf.keras.layers.Embedding(n_items, dim)
    
    def call(self, user_ids, pos_item_ids, neg_item_ids):
        u = self.user_emb(user_ids)
        pos_i = self.item_emb(pos_item_ids)
        neg_i = self.item_emb(neg_item_ids)
        
        pos_score = tf.reduce_sum(u * pos_i, axis=1)
        neg_score = tf.reduce_sum(u * neg_i, axis=1)
        
        # BPR loss: -log(sigmoid(pos_score - neg_score))
        loss = -tf.math.log(tf.nn.sigmoid(pos_score - neg_score) + 1e-8)
        return tf.reduce_mean(loss)

# Quick BPR training demo
bpr = BPRModel(n_users, n_items, 64)
bpr.compile(optimizer=tf.keras.optimizers.Adam(0.01))

# Generate BPR triples
bpr_data = []


## 10. Production Serving Architecture

| Component | Technology | Latency | Scale |
|-----------|------------|---------|-------|
| **Retrieval** | Faiss HNSW / ScaNN | <5ms | 10M+ items |
| **Ranking** | TF/TFRS / TensorRT | <20ms | 100-1000 candidates |
| **Feature Store** | Feast / Tecton | <2ms | Real-time features |
| **AB Testing** | Custom / MLflow | - | Multi-model |
| **Monitoring** | Evidently / Prometheus | - | Drift, latency |

**Key Takeaway**: Two-tower retrieval + Faiss ANN + neural ranking is the industry standard for large-scale recommenders. TFRS provides clean APIs; Faiss provides production-grade ANN. Cold start requires content features or popularity fallback. Always evaluate with ranking metrics (NDCG) not just retrieval metrics (Recall).